# NB27 - Real Retain Set (StreaksYolo) + trustworthy VAL

Buoc #2: dung StreaksYoloDataset (Zenodo 14047944) lam RETAIN SET THAT (streak that co nhan)
de (a) thay pseudo-clean nhiem, (b) dung VAL dang tin chot tham so.
Nen tang cho NB28 (#1 DINOv2 training-loss + #3 feature-suppression).


## 1. Setup & Installation


In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 7.3 MB/s eta 0:00:00


## 2. Imports and Configuration


In [2]:
import copy
import json
import logging
import os
import pickle
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog,
    DatasetMapper,
    MetadataCatalog,
    build_detection_train_loader,
    detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.structures import Boxes, Instances

logging.getLogger("detectron2").setLevel(logging.ERROR)
logging.getLogger("fvcore").setLevel(logging.ERROR)

# Kaggle input paths
ROOT = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models"
POISONED_WEIGHTS = f"{ROOT}/poisoned_model/poisoned_model.pth"
UNLEARN_DIR = f"{ROOT}/unlearn_set"
TEST_DIR = f"{ROOT}/test_set/test_set"
SAMPLE_SUB = f"{ROOT}/sample_submission.csv"

# Output paths
OUTPUT_DIR = "/kaggle/working/unlearned"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Model Architecture Configuration (Must match poisoned model exactly)
BASE_CONFIG = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1
IMG_W = IMG_H = 1024
BATCH_SIZE = 4

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
set_seed()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cuda


## 3. Dataset Registration & Image Loading


In [3]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

# Register unlearn dataset
UNLEARN_DATASET = "unlearn"

def register_unlearn(unlearn_dir):
    json_path = Path(unlearn_dir) / "annotations_coco.json"
    with open(json_path) as f:
        coco = json.load(f)
    dicts = [
        {
            "file_name": str(Path(unlearn_dir) / im["file_name"]),
            "height":    im["height"],
            "width":     im["width"],
            "image_id":  im["id"],
            "annotations": [],   # empty annotations for unlearning
        }
        for im in coco["images"]
    ]
    if UNLEARN_DATASET in DatasetCatalog:
        DatasetCatalog.remove(UNLEARN_DATASET)
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=["object"])
    print(f"Registered unlearn dataset: {len(dicts)} images (empty annotations)")
    return dicts

unlearn_dicts = register_unlearn(UNLEARN_DIR)

# Load original coco annotations for pruning box coordinates
with open(Path(UNLEARN_DIR) / "annotations_coco.json") as f:
    coco_data = json.load(f)

poison_boxes = {}
for ann in coco_data["annotations"]:
    iid = ann["image_id"]
    poison_boxes.setdefault(iid, []).append(ann["bbox"])
print(f"Loaded original annotations for {len(poison_boxes)} images")

class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_image(dataset_dict["file_name"])
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict


Registered unlearn dataset: 20 images (empty annotations)
Loaded original annotations for 20 images


## 4. build_cfg / predictor


In [4]:
def build_cfg(weights=POISONED_WEIGHTS, score_thresh=0.05):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    return cfg

def build_predictor(weights, score_thresh=0.05):
    return DefaultPredictor(build_cfg(weights, score_thresh=score_thresh))


## 5. Local maCADD Metric


In [5]:
A_FACTOR = 10.0
IOU_THRESHOLDS = np.arange(0.2, 0.91, 0.1)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    ax1, ay1, ax2, ay2 = a[:, 0:1], a[:, 1:2], a[:, 2:3], a[:, 3:4]
    bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    iw = np.maximum(0, np.minimum(ax2, bx2) - np.maximum(ax1, bx1))
    ih = np.maximum(0, np.minimum(ay2, by2) - np.maximum(ay1, by1))
    inter = iw * ih
    area_a = np.maximum(0, ax2 - ax1) * np.maximum(0, ay2 - ay1)
    area_b = np.maximum(0, bx2 - bx1) * np.maximum(0, by2 - by1)
    union = area_a + area_b - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)

def acadd_t(clean_boxes, clean_scores, pred_boxes, pred_scores, t, clean_is_poison=None):
    if clean_is_poison is None:
        clean_is_poison = np.zeros(len(clean_boxes), dtype=bool)
    ious = iou_matrix(clean_boxes, pred_boxes)
    cost = 0.0
    matched_c, matched_p = set(), set()
    if ious.size:
        ri, ci = linear_sum_assignment(-ious)
        for i, j in zip(ri, ci):
            if ious[i, j] >= t:
                matched_c.add(i); matched_p.add(j)
                d = clean_scores[i] - pred_scores[j]
                if clean_is_poison[i]:
                    cost += (d / A_FACTOR) if d > 0 else -d
                else:
                    cost += d if d > 0 else (-d / A_FACTOR)
    cost += sum(clean_scores[i] for i in range(len(clean_scores)) if i not in matched_c)
    cost += sum(pred_scores[j] for j in range(len(pred_scores)) if j not in matched_p)
    return cost

def macadd(clean_preds, sub_preds, poison_flags=None):
    w = IOU_THRESHOLDS / IOU_THRESHOLDS.sum()
    per_image = []
    for stem, (cb, cs) in clean_preds.items():
        keep = cs > 0.2
        cb, cs = cb[keep], cs[keep]
        pf = poison_flags.get(stem)[keep] if poison_flags else None
        pb, ps = sub_preds.get(stem, (np.zeros((0, 4)), np.zeros(0)))
        per_image.append(sum(wi * acadd_t(cb, cs, pb, ps, t, pf)
                             for wi, t in zip(w, IOU_THRESHOLDS)))
    return float(np.mean(per_image))


## 6. StreaksYolo retain loader (domain gap 640 JPEG 8-bit -> 1024)


In [6]:
import glob

# Path nguoi dung cung cap (uu tien), roi cac bien the mount pho bien, cuoi cung auto-detect.
PREFERRED_ROOTS = [
    "/kaggle/input/datasets/heon29/streaksyolodataset",
    "/kaggle/input/streaksyolodataset",
    "/kaggle/input/streaksyolodataset/StreaksYoloDataset",
]

def _has_labels(c):
    try:
        return c and Path(c).exists() and any(True for _ in Path(c).rglob("*.txt"))
    except Exception:
        return False

def _find_streaks_root():
    for c in PREFERRED_ROOTS:
        if _has_labels(c):
            return c
    cands = glob.glob("/kaggle/input/*streak*") + glob.glob("/kaggle/input/*yolo*") + glob.glob("/kaggle/input/*Streak*")
    for c in sorted(set(cands)):
        if _has_labels(c):
            return c
    return None

STREAKS_ROOT = _find_streaks_root()
print('STREAKS_ROOT =', STREAKS_ROOT)

def load_image_8bit(path, out=1024):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im is None:
        return None
    if im.ndim == 3:
        im = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0 * 255.0
    else:
        im = im.astype(np.float32)
    im = cv2.resize(im, (out, out), interpolation=cv2.INTER_LINEAR)
    im = np.clip(im, 0, 255).astype(np.float32)
    return np.repeat(im[:, :, None], 3, axis=2)

def _yolo_to_xyxy(line, W, H):
    p = line.split()
    if len(p) < 5:
        return None
    xc, yc, w, h = float(p[1])*W, float(p[2])*H, float(p[3])*W, float(p[4])*H
    return [xc - w/2, yc - h/2, xc + w/2, yc + h/2]

def build_retain_dicts(root, out=1024, limit=None):
    if root is None:
        return []
    imgs = []
    for ext in ('*.jpg','*.jpeg','*.png'):
        imgs += glob.glob(str(Path(root)) + '/**/' + ext, recursive=True)
    imgs = sorted(set(imgs))
    dicts = []
    for ip in imgs:
        stem = Path(ip).stem
        lbls = glob.glob(str(Path(root)) + '/**/' + stem + '.txt', recursive=True)
        if not lbls:
            continue
        boxes = []
        for ln in open(lbls[0]):
            ln = ln.strip()
            if ln:
                b = _yolo_to_xyxy(ln, out, out)
                if b:
                    boxes.append(b)
        if not boxes:
            continue
        dicts.append({'file_name': ip, 'boxes': np.array(boxes, np.float32)})
        if limit and len(dicts) >= limit:
            break
    return dicts

retain_dicts = build_retain_dicts(STREAKS_ROOT)
print(f'retain images voi >=1 streak box: {len(retain_dicts)}')
if retain_dicts:
    nb = np.mean([len(d['boxes']) for d in retain_dicts])
    print(f'avg streak box/anh: {nb:.2f} | vd box[0]: {retain_dicts[0]["boxes"][0].round(1).tolist()}')


STREAKS_ROOT = /kaggle/input/datasets/heon29/streaksyolodataset
retain images voi >=1 streak box: 1642
avg streak box/anh: 1.00 | vd box[0]: [140.8000030517578, 371.20001220703125, 782.4000244140625, 932.7999877929688]


## 7. Do retention tren streak THAT (poisoned model) - VAL dang tin


In [7]:
RETAIN_EVAL_N = min(60, len(retain_dicts))
if RETAIN_EVAL_N == 0:
    print('!! Chua co retain data. Upload StreaksYolo lam Kaggle dataset roi Add Data.')
else:
    pois = build_predictor(POISONED_WEIGHTS, score_thresh=0.05)
    hit = tot = 0
    conf_on_real = []
    for d in tqdm(retain_dicts[:RETAIN_EVAL_N], desc='retain eval'):
        im = load_image_8bit(d['file_name'])
        if im is None:
            continue
        out = pois(im)['instances'].to('cpu')
        pb = out.pred_boxes.tensor.numpy(); ps = out.scores.numpy()
        gt = d['boxes']
        tot += len(gt)
        if len(pb) and len(gt):
            iou = iou_matrix(gt, pb)
            for i in range(len(gt)):
                j = iou[i].argmax()
                if iou[i, j] >= 0.3:
                    hit += 1
                    conf_on_real.append(float(ps[j]))
    rec = hit / max(tot, 1)
    cr = np.array(conf_on_real) if conf_on_real else np.zeros(1)
    print(f'POISONED model tren streak THAT: recall@IoU0.3 = {rec:.3f} ({hit}/{tot})')
    print(f'conf tren streak that: med={np.median(cr):.3f} p25={np.percentile(cr,25):.3f} p75={np.percentile(cr,75):.3f}')
    print('-> VAL that + calib nguong; so conf med voi test (p75~0.324) de phat hien VAL synthetic lech.')


retain eval:   0%|          | 0/60 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
retain eval: 100%|██████████| 60/60 [00:07<00:00,  8.26it/s]

POISONED model tren streak THAT: recall@IoU0.3 = 0.317 (19/60)
conf tren streak that: med=0.336 p25=0.147 p75=0.363
-> VAL that + calib nguong; so conf med voi test (p75~0.324) de phat hien VAL synthetic lech.


## 8. Chan doan kich thuoc box: StreaksYolo vs poison vs model-on-test (do, khong doan)


In [8]:
# So kich thuoc box (max-dim = max(w,h)) giua 3 nguon de quyet dinh loc retain.
def _wh_stats(whs, name):
    whs = np.asarray(whs, np.float32)
    if len(whs) == 0:
        print(f"{name}: 0 box"); return
    md_ = np.stack([whs[:,0], whs[:,1]], 1)
    maxd = md_.max(1); area = whs[:,0]*whs[:,1]
    print(f"{name}: n={len(whs)} | max-dim med={np.median(maxd):.0f} "
          f"p25={np.percentile(maxd,25):.0f} p75={np.percentile(maxd,75):.0f} "
          f"p95={np.percentile(maxd,95):.0f} | area_med={np.median(area):.0f}")

# 1) StreaksYolo retain boxes (da o toa do 1024)
sy_wh = []
for d in retain_dicts:
    for x1,y1,x2,y2 in d["boxes"]:
        sy_wh.append([x2-x1, y2-y1])
_wh_stats(sy_wh, "StreaksYolo retain")

# 2) Poison boxes da biet (unlearn coco, toa do goc 1024)
po_wh = []
for anns in poison_boxes.values():
    for x,y,w,h in anns:
        po_wh.append([w, h])
_wh_stats(po_wh, "Poison (unlearn 20)")

# 3) Box model DOC bat tren mau test (dai dien phan bo streak/poison test)
pois = build_predictor(POISONED_WEIGHTS, score_thresh=0.2)
test_files = sorted(Path(TEST_DIR).glob("*.png"))[:120]
md_wh = []
for p in tqdm(test_files, desc="test box size"):
    out = pois(load_image(p))["instances"].to("cpu")
    b = out.pred_boxes.tensor.numpy()
    for x1,y1,x2,y2 in b:
        md_wh.append([x2-x1, y2-y1])
_wh_stats(md_wh, "Model-on-test (conf>0.2)")

# Goi y dai loc retain: giu box StreaksYolo co max-dim nam trong [p5,p95] cua model-on-test
if md_wh:
    mdd = np.asarray(md_wh, np.float32).max(1)
    lo, hi = np.percentile(mdd,5), np.percentile(mdd,95)
    syd = np.asarray(sy_wh, np.float32).max(1) if sy_wh else np.zeros(0)
    inb = ((syd>=lo)&(syd<=hi)).mean() if len(syd) else 0
    print(f"\n-> Dai model-on-test max-dim [p5,p95] = [{lo:.0f},{hi:.0f}] px")
    print(f"   StreaksYolo box nam trong dai nay: {100*inb:.1f}%")
    print("   -> Neu ty le thap = StreaksYolo lech kich thuoc; loc giu box trong dai truoc khi dung retain.")


StreaksYolo retain: n=1642 | max-dim med=480 p25=323 p75=653 p95=870 | area_med=72581
Poison (unlearn 20): n=20 | max-dim med=48 p25=31 p75=53 p95=59 | area_med=785


test box size: 100%|██████████| 120/120 [00:14<00:00,  8.25it/s]

Model-on-test (conf>0.2): n=139 | max-dim med=47 p25=35 p75=56 p95=81 | area_med=878

-> Dai model-on-test max-dim [p5,p95] = [26,81] px
   StreaksYolo box nam trong dai nay: 0.0%
   -> Neu ty le thap = StreaksYolo lech kich thuoc; loc giu box trong dai truoc khi dung retain.
